# 07. 직접 써 보기

문장을 넣으면 추천 결과와 근거를 보여준다. 실험용이 아니라 사용용 노트북이다.

사용법
1. 아래 **설정** 셀에서 필요하면 값을 바꾼다 (반환 개수, 텍스트 구성, 프랜차이즈 포함 여부).
2. **준비** 셀을 한 번 실행한다. 모델을 내려받아 두었으면 10초 안팎 걸린다.
3. 맨 아래 **질의** 셀에서 문장을 바꿔 가며 실행한다. `show("...")` 한 줄이면 된다.

결과 읽는 법
- 조건: 문장에서 뽑은 필수 조건(걸러냄), 선호 조건(점수), 언급·제외 메뉴, 처리하지 못한 표현.
- 점수: 정렬용 값이며 확률이 아니다. 조건이 있는 질의는 0.85 안팎, 없는 질의는 0.55 안팎으로 나온다.
- 라벨: 저장된 모델 추정 라벨이다. 추천 근거는 이 라벨을 기준으로 적힌다.

## 설정

In [1]:
TOP_K = 3                 # 반환 개수
TEXT_VARIANT = "B"        # "A": 음식명+분류, "B": A + 속성 라벨
INCLUDE_FRANCHISE = False  # True면 프랜차이즈 메뉴(업체명 있음)도 후보에 넣는다

## 준비 (한 번만 실행)

In [2]:
import sys
import time
from dataclasses import replace
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import E5Embedder
from src.preprocessing import parse_query, support_table
from src.recommendation import FULL, Recommender
from src.retrieval import load_index

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

t0 = time.time()
index, ref = load_index(TEXT_VARIANT, include_franchise=INCLUDE_FRANCHISE)
embedder = E5Embedder()
rec = Recommender(index, lambda text: embedder.encode_queries([text], show_progress=False)[0], ref,
                  replace(FULL, top_k=TOP_K))
print(f"준비 완료 {time.time() - t0:.1f}초: 텍스트 {TEXT_VARIANT}, 후보 {index.size}건"
      f"{'' if INCLUDE_FRANCHISE else ' (프랜차이즈 제외)'}, 장치 {embedder.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

준비 완료 9.3초: 텍스트 B, 후보 1123건 (프랜차이즈 제외), 장치 mps


/Users/jack/project/Menu-recommend-algorithmn/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


## 보기 함수

In [3]:
COLUMNS = ["순위", "메뉴명", "대표식품명", "주요라벨", "최종점수", "추천근거"]


def describe(parsed):
    lines = []
    if parsed.hard:
        lines.append("필수: " + "; ".join(f"{c.attribute}∈{'/'.join(c.allowed)} ← {c.evidence}" for c in parsed.hard))
    if parsed.soft:
        lines.append("선호: " + "; ".join(f"{c.attribute}∈{'/'.join(c.allowed)} ← {c.evidence}" for c in parsed.soft))
    if parsed.menu_terms:
        lines.append("언급 메뉴: " + ", ".join(parsed.menu_terms))
    if parsed.menu_exclusions:
        lines.append("제외 메뉴: " + ", ".join(e["term"] for e in parsed.menu_exclusions))
    if parsed.unhandled:
        lines.append("처리 못함: " + "; ".join(f"'{u['expression']}' ({u['reason']})" for u in parsed.unhandled))
    if parsed.ignored:
        lines.append("무시: " + ", ".join(i["expression"] for i in parsed.ignored))
    if parsed.contradictions:
        lines.append("모순: " + "; ".join(f"{c['attribute']} ({c['evidence']})" for c in parsed.contradictions))
    return "\n".join(lines) or "조건 없음 (임베딩 유사도만 사용)"


def show(text, k=None, details=False):
    """문장 하나에 대한 추천. details=True면 걸러진 항목 수와 중복·상한으로 빠진 항목도 보여준다"""
    result = rec.recommend(text, replace(rec.config, top_k=k or TOP_K))
    print(f"질의: {text!r}")
    print(describe(parse_query(text)))
    if result["상태"] != "ok":
        print(f"상태: {result['상태']} — {result['사유']}")
    if details:
        print(f"검색범위 {result['검색범위']}, 필터 통과 {result['필터통과']}/{result['후보수']}, "
              f"제외 사유 {result['필터제외사유']}")
        if result["제외"]:
            print("중복·상한으로 빠짐: " + ", ".join(f"{e['메뉴명']}({e['제외사유']})" for e in result["제외"][:8]))
    return pd.DataFrame(result["추천"], columns=COLUMNS)


def show_many(texts, k=None):
    """여러 문장을 한 표로. 메뉴명만 간단히 본다"""
    rows = []
    for text in texts:
        result = rec.recommend(text, replace(rec.config, top_k=k or TOP_K))
        rows.append({"질의": text, "상태": result["상태"],
                     **{f"{i + 1}위": it["메뉴명"] for i, it in enumerate(result["추천"])}})
    return pd.DataFrame(rows)

## 예시

In [4]:
show("맵지 않고 따뜻한 음식")

질의: '맵지 않고 따뜻한 음식'
필수: 매운맛∈없음 ← 맵지 않고
선호: 제공온도∈뜨거움/따뜻함 ← 따뜻한


,순위,메뉴명,대표식품명,주요라벨,최종점수,추천근거
0,1,화양적,화양적,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 부침, 기름짐 보통",0.8885,유사도 0.8407 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)
1,2,무 된장국,무 된장국,"매운맛 없음, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음, 든든함 가벼움",0.8878,유사도 0.8398 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=뜨거움 일치(따뜻한)
2,3,햄버거,햄버거,"매운맛 없음, 국물없음, 제공온도 따뜻함, 조리법 혼합, 기름짐 높음, 든든함",0.8878,유사도 0.8397 / 필수 매운맛=없음 충족(맵지 않고) / 선호 제공온도=따뜻함 일치(따뜻한)


In [5]:
show("피자 말고 얼큰한 국물", details=True)

질의: '피자 말고 얼큰한 국물'
선호: 매운맛∈보통/강함 ← 얼큰한; 제공온도∈뜨거움/따뜻함 ← 얼큰한; 국물∈국물요리 ← 국물
제외 메뉴: 피자
검색범위 [100], 필터 통과 94/100, 제외 사유 {'메뉴=피자': 6}


,순위,메뉴명,대표식품명,주요라벨,최종점수,추천근거
0,1,김치 콩나물국,김치 콩나물국,"매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 낮음",0.8775,유사도 0.8250 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
1,2,알탕,알탕,"매운맛 강함, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함 보통",0.8757,유사도 0.8224 / 선호 매운맛=강함 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)
2,3,해장국 뼈다귀,해장국,"매운맛 보통, 국물요리, 제공온도 뜨거움, 조리법 끓임, 기름짐 보통, 든든함",0.8755,유사도 0.8222 / 선호 매운맛=보통 일치(얼큰한) / 선호 제공온도=뜨거움 일치(얼큰한) / 선호 국물=국물요리 일치(국물)


In [6]:
show_many(["비 오는 날 얼큰한 국물 먹고 싶어", "차가운 면 요리", "튀김 말고 구운 고기", "안 매운 건 싫어"])

,질의,상태,1위,2위,3위
0,비 오는 날 얼큰한 국물 먹고 싶어,ok,해장국 뼈다귀,꽃게 매운탕,알탕
1,차가운 면 요리,ok,냉면 열무냉면,냉면 회냉면 홍어,회냉면
2,튀김 말고 구운 고기,ok,소고기산적,오징어불고기,오리불고기
3,안 매운 건 싫어,ok,복 매운탕,아귀 매운탕,미음


## 질의

아래 문장을 바꿔 실행한다. `details=True`를 붙이면 걸러진 항목과 사유까지 본다.
지원하는 표현 목록은 `pd.DataFrame(support_table())`로 볼 수 있다. "꼭", "무조건"을 붙이면 선호가 필수가 되고,
"~ 말고", "~ 싫어"는 제외가 된다. 날씨·기분 표현은 조건으로 쓰지 않고 임베딩에만 반영된다.

In [7]:
show("여기에 먹고 싶은 걸 적어 보세요, 예: 국물 없는 매운 음식")

질의: '여기에 먹고 싶은 걸 적어 보세요, 예: 국물 없는 매운 음식'
필수: 국물∈국물없음 ← 국물 없는
선호: 매운맛∈보통/강함 ← 매운


,순위,메뉴명,대표식품명,주요라벨,최종점수,추천근거
0,1,쟁반국수,쟁반국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.9030,유사도 0.8614 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
1,2,막국수,막국수,"매운맛 보통, 국물없음, 제공온도 차가움, 조리법 끓임, 기름짐 낮음, 든든함 보통",0.9022,유사도 0.8603 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)
2,3,해물볶음,해물볶음,"매운맛 보통, 국물없음, 제공온도 따뜻함, 조리법 볶음, 기름짐 보통, 든든함 보통",0.9019,유사도 0.8598 / 필수 국물=국물없음 충족(국물 없는) / 선호 매운맛=보통 일치(매운)


In [8]:
# 지원 표현 목록
pd.DataFrame(support_table())[["종류", "규칙", "예시", "조건"]]

,종류,규칙,예시,조건
0,unhandled,이중부정,안 매운 건 싫어,-
1,unhandled,허용표현,"매운 것도 괜찮아, 매워도 돼",-
2,unhandled,시원한국물,시원한 국물,-
3,unhandled,스키마외맛,단짠단짠한 음식,-
4,hard,매운맛_강함제외,"너무 맵지 않은, 너무 매운 거 싫어",매운맛∈없음/약함/보통
5,hard,매운맛_제외,"맵지 않은, 안 매운, 매운 거 싫어",매운맛∈없음
6,hard,국물_제외,"국물 없는, 국물이 없는, 국물 빼고",국물∈국물없음
7,hard,뜨거움_제외,뜨겁지 않은,제공온도∈따뜻함/상온/차가움
8,hard,차가움_제외,차갑지 않은,제공온도∈뜨거움/따뜻함/상온
9,hard,기름짐_제외,"느끼하지 않은, 기름기 적은",기름짐∈낮음/보통
